# AIC 2026 - Kaggle frame extraction smoke

Attach these datasets before running:

- `lyduchoang/aic-26-video` for raw videos
- `khoalequangminh/aic-test-dataset` for metadata/map files

This notebook only clones/pulls the repo branch and runs the offline frame extraction CLI for one video. It does not run TransNetV2, DAM, OCR, ASR, embeddings, or tests.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git"
BRANCH = "feat/offline-frame-extraction-transnetv2"
TARGET = Path("/kaggle/working/AIC-2026")

def run(command, *, cwd=None, env=None):
    print("$", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

clone_env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("AIC_GITHUB_TOKEN")
except Exception:
    token = None

if token:
    clone_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Bearer {token}",
    })

if TARGET.exists() and not (TARGET / ".git").is_dir():
    raise RuntimeError(f"Target exists but is not a git repo: {TARGET}")
if not TARGET.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(TARGET)], env=clone_env)
else:
    run(["git", "fetch", "origin", BRANCH], cwd=TARGET, env=clone_env)
    run(["git", "switch", BRANCH], cwd=TARGET, env=clone_env)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=TARGET, env=clone_env)

os.chdir(TARGET)
print("Repo:", Path.cwd())
run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=TARGET)
run(["git", "rev-parse", "--short", "HEAD"], cwd=TARGET)


In [ ]:
from collections import Counter
from pathlib import Path
import os
import subprocess

os.environ["AIC_DATA_ROOT"] = "/kaggle/input"
os.environ["AIC_ARTIFACT_ROOT"] = "/kaggle/working/aic2026-artifacts"
os.environ["AIC_VIDEO_ID"] = "L21_V001"

input_root = Path(os.environ["AIC_DATA_ROOT"])
print("AIC_DATA_ROOT:", input_root)
print("AIC_ARTIFACT_ROOT:", os.environ["AIC_ARTIFACT_ROOT"])
print("AIC_VIDEO_ID:", os.environ["AIC_VIDEO_ID"])

print("\nGPU preflight:")
try:
    gpu_report = subprocess.run(
        ["nvidia-smi", "-L"],
        check=False,
        capture_output=True,
        text=True,
    )
    print(gpu_report.stdout.strip() or gpu_report.stderr.strip() or "nvidia-smi returned no output")
    if gpu_report.returncode == 0:
        gpu_lines = [line for line in gpu_report.stdout.splitlines() if line.strip()]
        t4_lines = [line for line in gpu_lines if "T4" in line.upper()]
        if len(t4_lines) != 2:
            print(f"WARNING: expected Kaggle GPU T4 x2, detected {len(t4_lines)} T4 GPU(s).")
except FileNotFoundError:
    print("WARNING: nvidia-smi is not available. Check Kaggle Settings > Accelerator = GPU T4 x2.")

def print_tree(root: Path, *, max_depth: int = 3, max_items: int = 200):
    shown = 0
    for path in sorted(root.rglob("*"), key=lambda p: p.as_posix().lower()):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        indent = "  " * (depth - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}{rel.name}{suffix}")
        shown += 1
        if shown >= max_items:
            print(f"... truncated after {max_items} items")
            break

print_tree(input_root)
suffix_counts = Counter(path.suffix.lower() or "<none>" for path in input_root.rglob("*") if path.is_file())
print("Suffix counts:")
for suffix, count in suffix_counts.most_common(20):
    print(f"  {suffix}: {count}")

video_suffixes = {".avi", ".mkv", ".mov", ".mp4", ".webm"}
videos = sorted([p for p in input_root.rglob("*") if p.is_file() and p.suffix.lower() in video_suffixes])
print("First video candidates:")
for path in videos[:20]:
    print(" ", path)


In [ ]:
import json
import os
import subprocess
from pathlib import Path

artifact_root = Path(os.environ["AIC_ARTIFACT_ROOT"])
video_id = os.environ["AIC_VIDEO_ID"]
command = [
    "python",
    "scripts/extract_frame_samples.py",
    "--config",
    "configs/offline/frame_extraction.yaml",
    "--video-id",
    video_id,
    "--search-root",
    os.environ["AIC_DATA_ROOT"],
    "--output-root",
    str(artifact_root),
    "--limit",
    "10",
    "--resume",
]
print("$", " ".join(command))
result = subprocess.run(command, check=True, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

report = json.loads(result.stdout)
manifest_path = Path(report["output"])
print("Manifest:", manifest_path)
print("Frames:", report.get("frames"))


In [ ]:
import json
import os
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import Markdown, display

artifact_root = Path(os.environ["AIC_ARTIFACT_ROOT"])
manifest_path = artifact_root / "frame_extraction" / "manifests" / f"{os.environ['AIC_VIDEO_ID']}.jsonl"
records = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]

def timecode(seconds: float) -> str:
    total_ms = round(float(seconds) * 1000)
    ms = total_ms % 1000
    total_s = total_ms // 1000
    s = total_s % 60
    total_m = total_s // 60
    m = total_m % 60
    h = total_m // 60
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"

print("Records:", len(records))
print("Manifest:", manifest_path)
print("\nFull frame table:")
print("sample_n\ttimecode\tpts_time_s\tframe_idx\tfps\tkeyframe_n\tsource\timage_path")
for record in records:
    image_path = artifact_root / record["frame_relpath"]
    print(
        f"{record['sample_n']}\t{timecode(record['pts_time_s'])}\t"
        f"{record['pts_time_s']:.6f}\t{record['frame_idx']}\t{record['fps']:.4f}\t"
        f"{record.get('keyframe_n')}\t{record['sampling_source']}\t{image_path}"
    )

thumbs = []
for record in records:
    image_path = artifact_root / record["frame_relpath"]
    image = Image.open(image_path).convert("RGB")
    thumb = image.copy()
    thumb.thumbnail((240, 135))
    canvas = Image.new("RGB", (240, 160), "white")
    canvas.paste(thumb, ((240 - thumb.width) // 2, 0))
    draw = ImageDraw.Draw(canvas)
    draw.text(
        (8, 140),
        f"#{record['sample_n']} {timecode(record['pts_time_s'])}",
        fill=(0, 0, 0),
    )
    thumbs.append(canvas)

cols = 2
rows = (len(thumbs) + cols - 1) // cols
sheet = Image.new("RGB", (cols * 240, max(1, rows) * 160), "white")
for index, thumb in enumerate(thumbs):
    sheet.paste(thumb, ((index % cols) * 240, (index // cols) * 160))
display(Markdown("## Thumbnail overview"))
display(sheet)

display(Markdown("## Full extracted frames"))
for record in records:
    image_path = artifact_root / record["frame_relpath"]
    display(Markdown(
        f"### Frame #{record['sample_n']} - {timecode(record['pts_time_s'])} "
        f"({record['pts_time_s']:.3f}s, frame_idx={record['frame_idx']})"
    ))
    display(Image.open(image_path).convert("RGB"))
